# Profiling notebook for generic volume data

In [ ]:
%load_ext line_profiler
%load_ext autoreload
%autoreload 2

In [ ]:
import timeit
import numpy as np
from PIL import Image
import numpy.lib.recfunctions as rf
import os
os.add_dll_directory("C:/Program Files/NVIDIA GPU Computing Toolkit/CUDA/v12.9/bin")
from _preprocess_module import HeightFieldExtractor
from smooth_generic_volume_container import SmoothGenericVolumeContainer
from generic_volume_container import GenericVolumeContainer
import create_volume

In [ ]:
VOLUME = create_volume.create_huge_volume()

In [ ]:
def main():

    preprocessor = HeightFieldExtractor((create_volume.RESOLUTION_Y, create_volume.RESOLUTION_X), 2, 256)
    preprocessor.add_volume(VOLUME.container, (create_volume.RESOLUTION_X, create_volume.RESOLUTION_Y, create_volume.RESOLUTION_Z), 0.001)

    extended_heightfield, normal_map = preprocessor.extract_data_representation( 0.0 )

    # Save the extended heightfields
    for z in range(extended_heightfield.shape[2]):
        # entry_0 = rf.structured_to_unstructured(extended_heightfield[:,:,z]);
        entry = extended_heightfield[:,:,z]
        entry = entry.astype(np.uint16)
        img = Image.fromarray(entry, "I;16")
        img.save("output/integrated_"+str(z)+".tif")

    # Convert the first normal map
    normal_map = normal_map.squeeze(2)
    normal_map = rf.structured_to_unstructured( normal_map )
    normal_map = ( normal_map + 1.0 ) * 127.5
    normal = normal_map.astype(np.uint8)
    # print(normal)
    img = Image.fromarray(normal, "RGB")
    img.save("output/normal.tif")
    del preprocessor

In [ ]:
# for _ in range(3):
#     main()
# %timeit main()
%lprun -f main main()

In [ ]:
pp = None

def init_volume() -> None:
    global pp
    pp = HeightFieldExtractor((create_volume.RESOLUTION_Y, create_volume.RESOLUTION_X), 2, 256)
    pp.add_volume(create_volume.create_huge_volume().container, (create_volume.RESOLUTION_X, create_volume.RESOLUTION_Y, create_volume.RESOLUTION_Z), 0.001)

def tear_down():
    global pp
    del pp

def run():
    global pp
    pp.extract_data_representation( 0.0 )


In [ ]:
init_volume()

for _ in range(3):
    run()

%timeit run()

tear_down()